# Lookups and Collection Complexity

A lookup finds a value, a position, or a record. Its cost depends on the collection, whether a key or index is available, whether the data is sorted, and whether the structure must also support frequent insertion or deletion.

Big O notation describes how work grows as the number of values `n` grows. It describes growth rather than an exact execution time.

## 1. Common Big O Growth

| Complexity | Name | Growth |
|---|---|---|
| `O(1)` | Constant | Work does not grow with collection size |
| `O(log n)` | Logarithmic | Each step removes a large part of the remaining search space |
| `O(n)` | Linear | Work can grow in direct proportion to collection size |
| `O(n log n)` | Linearithmic | Typical comparison-based sorting cost |
| `O(n²)` | Quadratic | Often produced by comparing every value with every other value |

## 2. List Index Access: `O(1)`

A list stores references in a contiguous array. Python can calculate the location of an index directly, so accessing `items[index]` is constant time.

In [ ]:
products = ["Laptop", "Mouse", "Keyboard", "Monitor"]

print(products[0])
print(products[2])
print(products[-1])

## 3. List Value Search: `O(n)`

A list has no value-based index. Membership, `index`, `count`, and `remove` may scan values from left to right.

In [ ]:
product_skus = ["P100", "P200", "P300", "P400", "P500"]

print("P400" in product_skus)
print(product_skus.index("P400"))
print(product_skus.count("P200"))

## 4. Finding a Record in a List: `O(n)`

A list of dictionaries is convenient for ordered records, but finding one record by SKU requires a scan until a match is found.

In [ ]:
products = [
    {"sku": "P100", "name": "Laptop", "price": 65000},
    {"sku": "P200", "name": "Mouse", "price": 799},
    {"sku": "P300", "name": "Keyboard", "price": 1499},
    {"sku": "P400", "name": "Monitor", "price": 12999},
]

target_sku = "P300"
product = next((item for item in products if item["sku"] == target_sku), None)
print(product)

## 5. Tuple Lookups

A tuple has the same main lookup costs as a list: index access is `O(1)`, while searching by value is `O(n)`. Immutability does not make value search constant time.

In [ ]:
order_record = (1001, "Asha", 2499.0, "paid")
order_statuses = ("paid", "shipped", "cancelled", "delivered")

print(order_record[1])
print("cancelled" in order_statuses)
print(order_statuses.index("delivered"))

## 6. Dictionary Key Lookup: Average `O(1)`

A dictionary uses a hash table. A key's hash leads to its expected storage location without scanning every entry. Lookup is `O(1)` on average and can degrade to `O(n)` in the worst case.

In [ ]:
products_by_sku = {
    "P100": {"name": "Laptop", "price": 65000},
    "P200": {"name": "Mouse", "price": 799},
    "P300": {"name": "Keyboard", "price": 1499},
    "P400": {"name": "Monitor", "price": 12999},
}

print(products_by_sku["P300"])
print(products_by_sku.get("P999"))

## 7. Dictionary Value Search: `O(n)`

Fast lookup applies to keys. Searching dictionary values still requires scanning the values unless a reverse index is maintained.

In [ ]:
product_names_by_sku = {"P100": "Laptop", "P200": "Mouse", "P300": "Keyboard"}

print("Mouse" in product_names_by_sku.values())
matching_sku = next(
    (sku for sku, name in product_names_by_sku.items() if name == "Mouse"),
    None,
)
print(matching_sku)

## 8. Set Membership: Average `O(1)`

A set also uses hashing. It provides fast membership checks but does not map a key to a separate value.

In [ ]:
blocked_skus = {"P120", "P215", "P410"}

print("P215" in blocked_skus)
print("P100" in blocked_skus)

## 9. List and Set Membership Compared

List membership is `O(n)` because it scans. Set membership is `O(1)` on average because it hashes. Building the set costs `O(n)`, so conversion is most useful when many membership checks will follow.

In [ ]:
catalog_list = [f"SKU-{number:05d}" for number in range(10_000)]
catalog_set = set(catalog_list)
target_skus = ["SKU-00010", "SKU-05000", "SKU-09999", "SKU-12000"]

list_results = [sku in catalog_list for sku in target_skus]
set_results = [sku in catalog_set for sku in target_skus]

print(list_results)
print(set_results)

## 10. Unsorted Search: `O(n)`

Without ordering or a hash-based index, a search cannot safely discard unseen values. In the worst case it checks every element.

In [ ]:
unsorted_prices = [12999, 499, 65000, 1499, 799]
target_price = 65000

found_at = None
for index, price in enumerate(unsorted_prices):
    if price == target_price:
        found_at = index
        break

print(found_at)

## 11. Sorted Search With Binary Search: `O(log n)`

Binary search compares the middle value and discards half of the remaining range each step. The data must already be sorted according to the same key used by the search.

In [ ]:
from bisect import bisect_left

sorted_prices = [499, 799, 1499, 12999, 65000]
target_price = 12999
position = bisect_left(sorted_prices, target_price)
found = position < len(sorted_prices) and sorted_prices[position] == target_price

print(position)
print(found)

## 12. Binary Search on Sorted Records

A separate sorted key list provides `O(log n)` position lookup. Retrieving the record by that position is `O(1)`.

In [ ]:
products_sorted_by_sku = [
    {"sku": "P100", "name": "Laptop"},
    {"sku": "P200", "name": "Mouse"},
    {"sku": "P300", "name": "Keyboard"},
    {"sku": "P400", "name": "Monitor"},
]
sku_index = [product["sku"] for product in products_sorted_by_sku]

target_sku = "P300"
position = bisect_left(sku_index, target_sku)
product = (
    products_sorted_by_sku[position]
    if position < len(sku_index) and sku_index[position] == target_sku
    else None
)
print(product)

## 13. Sorting Cost: `O(n log n)`

Binary search is only `O(log n)` after the data is sorted. If unsorted data must be sorted for one search, the total is dominated by sorting at `O(n log n)`, making a direct `O(n)` scan cheaper for that single lookup.

In [ ]:
unsorted_skus = ["P400", "P100", "P300", "P200"]
sorted_skus = sorted(unsorted_skus)
position = bisect_left(sorted_skus, "P300")

print(unsorted_skus)
print(sorted_skus)
print(position)

## 14. Maintaining Sorted Order

`bisect.insort` finds an insertion position in `O(log n)`, but list insertion shifts later elements and is `O(n)`. The complete insertion therefore remains `O(n)`.

In [ ]:
from bisect import insort

sorted_prices = [499, 799, 1499, 12999, 65000]
insort(sorted_prices, 2499)
print(sorted_prices)

## 15. Finding the First Matching Value

`next` with a generator returns the first match without constructing a list. The lookup is still `O(n)` in the worst case.

In [ ]:
first_premium_product = next(
    (product for product in products if product["price"] >= 10000),
    None,
)
print(first_premium_product)

## 16. Finding All Matching Values

Finding every match requires visiting every record, so filtering is `O(n)`. Creating the result also uses space proportional to the number of matches.

In [ ]:
premium_products = [product for product in products if product["price"] >= 10000]
print(premium_products)

## 17. Building a Dictionary Index: `O(n)`

Building an index requires one pass. Afterward, each key lookup is `O(1)` on average. This is useful when many lookups will be performed against mostly stable data.

In [ ]:
products_by_sku = {product["sku"]: product for product in products}
requested_skus = ["P300", "P100", "P400", "P999"]
results = [products_by_sku.get(sku) for sku in requested_skus]

print(results)

## 18. Multiple Records Per Lookup Key

A dictionary can map one key to a list of matching records. Building the grouping index is `O(n)`, and locating the group is `O(1)` on average.

In [ ]:
orders = [
    {"order_id": 1001, "customer_id": "C101", "amount": 2499},
    {"order_id": 1002, "customer_id": "C102", "amount": 12999},
    {"order_id": 1003, "customer_id": "C101", "amount": 799},
    {"order_id": 1004, "customer_id": "C103", "amount": 5499},
]

orders_by_customer = {}
for order in orders:
    orders_by_customer.setdefault(order["customer_id"], []).append(order)

print(orders_by_customer.get("C101", []))

## 19. Composite-Key Lookup

A tuple can combine several fields into one hashable dictionary key. The lookup remains `O(1)` on average.

In [ ]:
inventory = {
    ("Chennai", "P100"): 12,
    ("Chennai", "P200"): 35,
    ("Bengaluru", "P100"): 8,
}

print(inventory[("Chennai", "P200")])

## 20. Minimum and Maximum

`min` and `max` scan an unsorted collection in `O(n)`. For a non-empty sorted list, accessing the first or last element is `O(1)`.

In [ ]:
unsorted_prices = [12999, 499, 65000, 1499, 799]
sorted_prices = sorted(unsorted_prices)

print(min(unsorted_prices))
print(max(unsorted_prices))
print(sorted_prices[0])
print(sorted_prices[-1])

## 21. Insertions and Deletions in Lists

Appending or popping at the end is `O(1)` amortized. Inserting or deleting near the beginning is `O(n)` because later elements must shift. Removing by value also includes an `O(n)` search.

In [ ]:
dispatch_orders = [1001, 1002, 1003]
dispatch_orders.append(1004)
last_order = dispatch_orders.pop()
dispatch_orders.insert(0, 1000)
first_order = dispatch_orders.pop(0)

print(last_order)
print(first_order)
print(dispatch_orders)

## 22. Queue Lookup and End Operations

A `deque` provides `O(1)` append and pop operations at both ends. Index access near the middle is `O(n)`, so a deque is not a replacement for random-access lists.

In [ ]:
from collections import deque

dispatch_queue = deque([1001, 1002, 1003])
dispatch_queue.append(1004)
first_order = dispatch_queue.popleft()

print(first_order)
print(dispatch_queue[0])
print(list(dispatch_queue))

## 23. Slicing Cost

A list or tuple slice copies references into a new collection. Its time and additional space are `O(k)`, where `k` is the number of selected values. Indexing one value remains `O(1)`.

In [ ]:
order_ids = list(range(1001, 1101))
single_order_id = order_ids[49]
page = order_ids[40:60]
copy_of_all_ids = order_ids[:]

print(single_order_id)
print(page)
print(len(copy_of_all_ids))

## 24. Set Operations

For sets `a` and `b`, union is generally `O(len(a) + len(b))`. Intersection is generally proportional to the smaller input. Difference is generally proportional to the left input. Exact behavior depends on operand form and implementation details.

In [ ]:
catalog_skus = {"P100", "P200", "P300", "P400"}
inventory_skus = {"P100", "P300", "P500"}

print(catalog_skus | inventory_skus)
print(catalog_skus & inventory_skus)
print(catalog_skus - inventory_skus)

## 25. Nested-Loop Lookup: `O(n × m)`

Searching one list for every value in another list can become quadratic when both lists grow together. Converting the lookup side to a set changes repeated membership checks to average `O(1)`.

In [ ]:
requested_skus = ["P100", "P300", "P500"]
available_skus = ["P100", "P200", "P300", "P400"]

matches_with_lists = [sku for sku in requested_skus if sku in available_skus]

available_sku_set = set(available_skus)
matches_with_set = [sku for sku in requested_skus if sku in available_sku_set]

print(matches_with_lists)
print(matches_with_set)

## 26. Complexity Summary

| Operation | List / Tuple | Dictionary | Set | Sorted List |
|---|---:|---:|---:|---:|
| Index access | `O(1)` | — | — | `O(1)` |
| Search by value | `O(n)` | `O(n)` over values | Average `O(1)` membership | `O(log n)` with binary search |
| Key lookup | — | Average `O(1)` | Average `O(1)` membership | — |
| Append at end | List: amortized `O(1)` | — | — | `O(n)` when order must remain sorted |
| Insert/delete at beginning | List: `O(n)` | — | — | `O(n)` |
| Add/delete by key or value | — | Average `O(1)` | Average `O(1)` | `O(n)` insertion/deletion |
| Minimum/maximum | `O(n)` | `O(n)` | `O(n)` | `O(1)` at an end |
| Full sort | `O(n log n)` | Sort selected keys/items: `O(n log n)` | Convert and sort: `O(n log n)` | Already maintained |
| Slice of `k` values | `O(k)` | — | — | `O(k)` |

## 27. Choosing a Lookup Structure

| Requirement | Suitable Structure |
|---|---|
| Retrieve by numeric position | List or tuple |
| Retrieve a record by unique SKU or order ID | Dictionary |
| Repeated membership checks | Set |
| Preserve duplicates and insertion order | List |
| Fixed record with positional fields | Tuple |
| Range queries and ordered traversal | Sorted list with `bisect` |
| Frequent insertions and removals at both ends | `deque` |
| Many records for one lookup key | Dictionary of lists |

## 28. Measuring Lookup Time

`timeit` repeats an operation and reports elapsed time. Measurements depend on hardware, Python version, input distribution, and collection size, while Big O describes the long-term growth pattern.

In [ ]:
from timeit import timeit

sku_list = [f"SKU-{number:05d}" for number in range(20_000)]
sku_set = set(sku_list)
sku_dict = {sku: index for index, sku in enumerate(sku_list)}
target = "SKU-19999"

list_time = timeit(lambda: target in sku_list, number=1_000)
set_time = timeit(lambda: target in sku_set, number=1_000)
dict_time = timeit(lambda: target in sku_dict, number=1_000)

print(f"List membership      : {list_time:.6f} seconds")
print(f"Set membership       : {set_time:.6f} seconds")
print(f"Dictionary membership: {dict_time:.6f} seconds")

## 29. Complete Ecommerce Lookup Design

In [ ]:
product_records = [
    {"sku": "P100", "name": "Laptop", "price": 65000, "category": "Electronics"},
    {"sku": "P200", "name": "Mouse", "price": 799, "category": "Accessories"},
    {"sku": "P300", "name": "Keyboard", "price": 1499, "category": "Accessories"},
    {"sku": "P400", "name": "Monitor", "price": 12999, "category": "Electronics"},
]
blocked_skus = {"P300", "P900"}

products_by_sku = {product["sku"]: product for product in product_records}
products_by_category = {}
for product in product_records:
    products_by_category.setdefault(product["category"], []).append(product)

products_by_price = sorted(product_records, key=lambda product: product["price"])
price_index = [product["price"] for product in products_by_price]

requested_sku = "P200"
if requested_sku in blocked_skus:
    selected_product = None
else:
    selected_product = products_by_sku.get(requested_sku)

minimum_price = 1000
start = bisect_left(price_index, minimum_price)
products_at_or_above_price = products_by_price[start:]

print("SKU lookup:", selected_product)
print("Category lookup:", products_by_category["Accessories"])
print("Price-range lookup:", products_at_or_above_price)